# `EukaryoticToeholdGate` — usage example (trailing-Kozak layout)

A real, end-to-end run of the single-input eukaryotic toehold switch —
`EukaryoticToeholdGate` in `engine.gates.toehold`. This class is fully implemented
and tested (`tests/engine/gates/test_toehold.py`, 45 passing tests as of this
writing). This is a sibling to [`toehold.ipynb`](toehold.ipynb) (which drives the
gate through a stub `FoldEngine` for fast, dependency-free iteration and covers both
hosts generically) — here we build `EukaryoticToeholdGate` specifically and use the
**real** `FoldEngine` (ViennaRNA) throughout, so every number below is a genuine
fold, not a placeholder.

`ToeholdGate` builds two structurally different eukaryotic layouts (commits
`d754812`, `ee2f5f6`):

* **`"loop"`** — Kozak embedded in the hairpin loop, ported unmodified from the
  prokaryotic mechanism (steric occlusion of the start codon). Unvalidated for a
  eukaryotic toehold specifically — see `KOZAK_LAYOUTS`'s docstring. It also turns
  out Kozak and AUG are *not* adjacent in this layout (a 6 nt stem-closing segment
  sits between them), which breaks the Kozak consensus's own adjacency requirement.
* **`"trailing"`** — Kozak and the start codon sit *after* the closed hairpin
  instead, modelling scanning-ribosome blockage (docs/modalities.md) rather than
  direct start-codon occlusion. This is the layout the team's own eukaryotic
  scripts actually build (`plasmid_prefix + trg_bind_region + loop + stem_down +
  kozak` — Kozak last), and Kozak is genuinely adjacent to AUG here. It also carries
  no trailing `LINKER_SEQUENCE` — cap-dependent scanning initiates the instant the
  40S subunit meets Kozak+AUG, so nothing after the start codon matters to finding
  it, and the payload attaches directly.

It also optionally takes the real effector gene (`payload`, commit `74bf7b4`) and
folds its own first nucleotides into every design instead of a placeholder — because
the real downstream sequence can change which design actually scores best, not just
which one looks best in isolation.

This notebook builds the gate restricted to **`"trailing"`** only
(`kozak_layouts=("trailing",)`), pools designs across the **top 150 trigger
candidates** (not just the single best-scoring one), and reserves at least 40 of
those 150 from the 3&#8242; UTR (from 100 nt before the CDS ends to the end of the
transcript) so that region gets a real chance to compete rather than being crowded
out by however TriggerScorer's global ranking happens to fall.

No Django, no worker, no pipeline — just the gate class, constructed and called
directly, the way `pipeline.py` would use it internally.

## Setup

In [ ]:
# Put <repo>/src on the path. Search upward from cwd for pyproject.toml so this works
# wherever Jupyter is launched from.
import sys
from pathlib import Path

for _base in (Path.cwd(), *Path.cwd().parents):
    if (_base / "pyproject.toml").exists():
        _src = _base / "src"
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break

from engine.domain import Host, Regulation, SelectedGene, TriggerSet, Constraints
from engine.gates.toehold import EukaryoticToeholdGate
from engine.gates.tools.folding import FoldEngine
from engine.gates.tools.translation import TranslationScorer
from engine.gates.tools.codons import CodonOptimizer
from engine.stages.folding import FoldProfiler
from engine.stages.motifs import MotifScreener
from engine.stages.off_target import OffTargetScanner
from engine.stages.triggers import TriggerScorer
from engine import sequences as sq

## 1. Build the tools, once

Per `CLAUDE.md` §5: tools are constructed once and handed to the gate, never built
inside a stage or family. `FoldEngine`'s cache is only useful if every caller shares
one instance — a second `FoldEngine()` means a cold cache and, worse, a second
chance to fold at a different temperature.

In [ ]:
host = Host.HUMAN  # HUMAN | YEAST both take the eukaryotic (Kozak) track

folder = FoldEngine(temperature=37.0)
translation = TranslationScorer(host)
codons = CodonOptimizer(host)

# The real effector gene — from `eff` in CERNAL_FUNCTIONS.py (a GFP-family CDS, already
# RNA, 759 nt, tandem stop UAA UAA). When set, the gate folds its real first
# PAYLOAD_HEAD_LENGTH nt into every design instead of a placeholder — see
# PAYLOAD_HEAD_LENGTH's docstring (commit 74bf7b4) for why this can change which
# design scores best. Set to None to see the placeholder behaviour instead.
PAYLOAD_CDS = (
    "AUGCGUAAAGGAGAAGAACUUUUCACUGGAGUUGUCCCAAUUCUUGUUGAAUUAGAUGGUGAUGUUAAUGGGCACAAAUUUUCUGUCAG"
    "UGGAGAGGGUGAAGGUGAUGCAACAUACGGAAAACUUACCCUUAAAUUUAUUUGCACUACUGGAAAACUACCUGUUCCGUGGCCAACAC"
    "UUGUCACUACUUUCGGUUAUGGUGUUCAAUGCUUUGCGAGAUACCCAGAUCACAUGAAACAGCAUGACUUUUUCAAGAGUGCCAUGCCC"
    "GAAGGUUACGUACAGGAAAGAACUAUAUUUUUCAAAGAUGACGGGAACUACAAGACACGUGCUGAAGUCAAGUUUGAAGGUGAUACCCU"
    "UGUUAAUAGAAUCGAGUUAAAAGGUAUUGAUUUUAAAGAAGAUGGAAACAUUCUUGGACACAAAUUGGAAUACAACUAUAACUCACACA"
    "AUGUAUACAUCAUGGCAGACAAACAAAAGAAUGGAAUCAAAGUUAACUUCAAAAUUAGACACAACAUUGAAGAUGGAAGCGUUCAACUA"
    "GCAGACCAUUAUCAACAAAAUACUCCGAUUGGCGAUGGCCCUGUCCUUUUACCAGACAACCAUUACCUGUCCACACAAUCUGCCCUUUC"
    "GAAAGAUCCCAACGAAAAGAGAGACCACAUGGUCCUUCUUGAGUUUGUAACCGCUGCUGGGAUUACACAUGGCAUGGAUGAACUAUACA"
    "AAAGGCCUGCAGCAAACGACGAAAACUACGCUGCAUCAGUUUAAUAA"
)

# kozak_layouts restricts generate_designs to the "trailing" layout only — the default
# (omit this argument) sweeps both "loop" and "trailing" and lets engine.scoring rank
# across them. This mirrors how `host` is a constructor parameter rather than a
# subclass (docs/engine.md §2.4).
gate = EukaryoticToeholdGate(
    host, folder, translation, codons, kozak_layouts=("trailing",), payload=PAYLOAD_CDS
)
print(gate.required_tools())
print("kozak_layouts:", gate.kozak_layouts)
print("payload_head:", gate.payload_head)

## 2. Pick trigger candidates — via the real `TriggerScorer` (stage 2)

`TriggerScorer.score` is fully implemented (`tests/engine/test_triggers.py`, 47
passing tests): it slides every window of every length in
`constraints.trigger_lengths` across the transcript, screens out forbidden motifs,
folds each survivor for `openness`/`accessibility`/`mfe` via the same `FoldEngine`
the gate uses, checks off-targets, ranks by `accessibility * segment_specificity`,
and yields the top candidates per gene — so we use it for real here instead of
hand-picking one window.

`OffTargetScanner` itself is still a stub (`find_similar`/`scan_trigger` raise
`NotImplementedError`) — **except** when handed an empty transcriptome, which is a
deliberate early-return for exactly this case (a `direct` submission, or a demo like
this one, with no reference index to scan against): `scan_trigger` returns a clean
`OffTargetReport(hits=(), penalty=0.0)` rather than raising. That is a real,
documented behaviour of the class, not a workaround.

Earlier versions of this notebook carried only `candidates[0]`, then the top 5,
through — which never gave a weaker-looking trigger the chance to turn out to fold
into a better switch. This version carries the top 150 through instead, with a floor
on how many must come from the 3&#8242; UTR rather than leaving that to chance — see the
cell below.

In [ ]:
# The full-length human AREG mRNA, as cDNA/DNA notation (T, not U) — same alphabet trap
# CLAUDE.md warns about, so normalise with to_rna() before anything else touches it.
transcript_dna = (
    "AGACGTTCGCACACCTGGGTGCCAGCGCCCCAGAGGTCCCGGGACAGCCCGAGGCGCCGCGCCCGCCGCCCCGAGCTCCCC"
    "AAGCCTTCGAGAGCGGCGCACACTCCCGGTCTCCACTCGCTCTTCCAACACCCGCTCGTTTTGGCGGCAGCTCGTGTCCCA"
    "GAGACCGAGTTGCCCCAGAGACCGAGACGCCGCCGCTGCGAAGGACCAATGAGAGCCCCGCTGCTACCGCCGGCGCCGGTG"
    "GTGCTGTCGCTCTTGATACTCGGCTCAGGCCATTATGCTGCTGGATTGGACCTCAATGACACCTACTCTGGGAAGCGTGAA"
    "CCATTTTCTGGGGACCACAGTGCTGATGGATTTGAGGTTACCTCAAGAAGTGAGATGTCTTCAGGGAGTGAGATTTCCCCT"
    "GTGAGTGAAATGCCTTCTAGTAGTGAACCGTCCTCGGGAGCCGACTATGACTACTCAGAAGAGTATGATAACGAACCACAA"
    "ATACCTGGCTATATTGTCGATGATTCAGTCAGAGTTGAACAGGTAGTTAAGCCCCCCCAAAACAAGACGGAAAGTGAAAAT"
    "ACTTCAGATAAACCCAAAAGAAAGAAAAAGGGAGGCAAAAATGGAAAAAATAGAAGAAACAGAAAGAAGAAAAATCCATGT"
    "AATGCAGAATTTCAAAATTTCTGCATTCACGGAGAATGCAAATATATAGAGCACCTGGAAGCAGTAACATGCAAATGTCA"
    "GCAAGAATATTTCGGTGAACGGTGTGGGGAAAAGTCCATGAAAACTCACAGCATGATTGACAGTAGTTTATCAAAAATTG"
    "CATTAGCAGCCATAGCTGCCTTTATGTCTGCTGTGATCCTCACAGCTGTTGCTGTTATTACAGTCCAGCTTAGAAGACAA"
    "TACGTCAGGAAATATGAAGGAGAAGCTGAGGAACGAAAGAAACTTCGACAAGAGAATGGAAATGTACATGCTATAGCATA"
    "ACTGAAGATAAAATTACAGGATATCACATTGGAGTCACTGCCAAGTCATAGCCATAAATGATGAGTCGGTCCTCTTTCCA"
    "GTGGATCATAAGACAATGGACCCTTTTTGTTATGATGGTTTTAAACTTTCAATTGTCACTTTTTATGCTATTTCTGTATA"
    "TAAAGGTGCACGAAGGTAAAAAGTATTTTTTCAAGTTGTAAATAATTTATTTAATATTTAATGGAAGTGTATTTATTTTA"
    "CAGCTCATTAAACTTTTTTAACCAAA"
)
transcript = sq.to_rna(transcript_dna)
assert sq.is_valid_rna(transcript)
print(f"transcript length: {len(transcript)} nt")

## Locate the CDS, to define the 3&#8242; UTR

Nothing in this engine parses gene annotation (no GTF/GFF loader exists — the CSV/
sequence-database parser is one of the two things `CLAUDE.md` §7 names as having "no
home yet"). To split this transcript into CDS and 3&#8242; UTR at all, approximate the
CDS as the **longest open reading frame** — every AUG in every frame, paired with its
nearest downstream in-frame stop, keeping the longest. This is a heuristic standing in
for real annotation, not something to trust for a transcript whose true CDS isn't
already known by other means — flagged, not silently treated as ground truth.

In [ ]:
def longest_orf(sequence):
    """(start, stop_codon_start, length) of the longest AUG-to-in-frame-stop ORF,
    across all three frames. A heuristic CDS finder, not a gene-model parser."""
    best = None
    for frame in (0, 1, 2):
        augs = sq.find_augs(sequence, frame=frame)
        stops = sq.find_stops(sequence, frame=frame)
        for start in augs:
            downstream_stops = [s for s in stops if s > start]
            if not downstream_stops:
                continue
            stop = min(downstream_stops)
            length = stop - start
            if best is None or length > best[2]:
                best = (start, stop, length, frame)
    return best


orf_start, stop_codon_start, orf_length, orf_frame = longest_orf(transcript)
cds_end = stop_codon_start + 3  # exclusive, i.e. transcript[cds_end:] is the 3' UTR

print(f"longest ORF: {orf_start}-{stop_codon_start} ({orf_length} nt, frame {orf_frame})")
print(f"cds_end: {cds_end}  (transcript length {len(transcript)}, "
      f"3' UTR is {len(transcript) - cds_end} nt)")

In [ ]:
# Stage-2 tools, built once (same injection rule as the gate's own tools).
profiler = FoldProfiler()
screener = MotifScreener()
off_target = OffTargetScanner(transcriptome={})  # empty: no reference index for this demo
scorer = TriggerScorer(profiler, off_target, screener, folder)  # shares the gate's FoldEngine

# TriggerScorer.TOP_K_PER_GENE defaults to 50 — a search-budget knob (its own
# docstring), not a ceiling meaningful here. Raised so the 3' UTR quota below has the
# whole ranked pool to draw from, not just whatever survived a 50-candidate cutoff.
scorer.TOP_K_PER_GENE = 5000

# In a real run this comes from GeneSelector (stage 1); stand in with a minimal
# SelectedGene since this demo starts from a single known transcript.
gene = SelectedGene(
    gene_id="AREG",
    symbol="AREG",
    regulation=Regulation.UP,
    log2_fold_change=2.0,
    score=1.0,
)
constraints = Constraints(trigger_lengths=(30, 33, 36), max_switch_length=200)

candidates = list(scorer.score([gene], {"AREG": transcript}, constraints))
print(f"{len(candidates)} candidate(s) survived screening, best-scoring first\n")
for c in candidates[:5]:
    print(
        f"  {c.trigger_id:22} start={c.start_index:4} len={c.length:2}  "
        f"score={c.score:.3f}  accessibility={c.accessibility:.3f}  gc={c.gc_content:.1f}"
    )

# Select the top N_TRIGGERS, with a floor on how many come from the 3' UTR — a window
# starting anywhere from 100 nt before the CDS ends onward. Nothing in TriggerScorer
# enforces a region quota (it ranks globally by score alone), so this is done here,
# not in the engine: reserve MIN_FROM_UTR slots from the UTR-only pool, then fill the
# rest with whatever scores best overall (UTR leftovers included, so a strong UTR
# candidate isn't capped out of the general competition).
N_TRIGGERS = 150
MIN_FROM_UTR = 40
utr_start = cds_end - 100

utr_pool = [c for c in candidates if c.start_index >= utr_start]
orf_pool = [c for c in candidates if c.start_index < utr_start]
print(f"\n{len(utr_pool)} candidate(s) start in the 3' UTR (>= {utr_start}), "
      f"{len(orf_pool)} start in or before the CDS")

utr_reserved = utr_pool[:MIN_FROM_UTR]
remaining = sorted(utr_pool[MIN_FROM_UTR:] + orf_pool, key=lambda c: c.score, reverse=True)
top_triggers = sorted(
    utr_reserved + remaining[: N_TRIGGERS - len(utr_reserved)],
    key=lambda c: c.score,
    reverse=True,
)

n_utr_selected = sum(1 for c in top_triggers if c.start_index >= utr_start)
print(f"\nselected {len(top_triggers)} trigger(s): {n_utr_selected} from the 3' UTR "
      f"(floor was {MIN_FROM_UTR}), {len(top_triggers) - n_utr_selected} from the CDS")

## 3. Wrap each trigger in its own `TriggerSet`

`TriggerSet` is the circuit's inputs (one activator each here — every trigger builds
its own single-input switch, independently). `Constraints` were already built above,
since `TriggerScorer` needed them too — a run builds `Constraints` once from
`params["constraints"]` and threads the same object through every stage.

In [ ]:
trigger_sets = [TriggerSet(activators=(t,)) for t in top_triggers]

print(f"{len(trigger_sets)} TriggerSet(s) built, e.g.:")
for ts in trigger_sets[:3]:
    print(" ", ts.activators[0].trigger_id, "| arity:", ts.arity, "| logic:", ts.logic_type)

## 4. `is_compatible()` — cheap check before generating anything

Arity, host, trigger length window — nothing here folds. Checked per trigger set,
same as a real run would (a family is checked against every trigger set it might
build from).

In [ ]:
compatible_trigger_sets = []
incompatible = []
for ts in trigger_sets:
    compatibility = gate.is_compatible(ts, constraints)
    if compatibility.ok:
        compatible_trigger_sets.append(ts)
    else:
        incompatible.append((ts, compatibility))

print(f"{len(compatible_trigger_sets)}/{len(trigger_sets)} trigger set(s) compatible")
for ts, reason in incompatible[:5]:
    print(" incompatible:", ts.activators[0].trigger_id, reason)

assert compatible_trigger_sets, "no trigger set survived is_compatible"

## 5. `generate_designs()` — the full candidate pool

Called **once per compatible trigger set**, not just the top one — exactly how a real
run explores the search space, since a lower-ranked trigger can still fold into a
better switch. With `kozak_layouts=("trailing",)`, each trigger set yields one
`GateDesign` per `toehold_lengths` x `TRAILING_LOOP_LENGTHS` x `KOZAK_LINKER_LENGTHS`
combination it supports. Pooled together, this is the candidate set `engine.scoring`
would actually rank in a real run.

In [ ]:
designs = []
for ts in compatible_trigger_sets:
    designs.extend(gate.generate_designs(ts, constraints))

n_from_utr = sum(
    1 for d in designs if d.trigger_set.activators[0].start_index >= utr_start
)
print(f"{len(designs)} design(s) across {len(compatible_trigger_sets)} trigger(s)")
print(f"  {n_from_utr} design(s) trace back to a 3' UTR trigger")
print(f"  {len(designs) - n_from_utr} design(s) trace back to a CDS-region trigger")
print("\nfirst few:")
for d in designs[:5]:
    print(
        f"  {d.design_id:52} {d.length:3} nt  "
        f"trigger={d.trigger_set.activators[0].trigger_id:18}  "
        f"loop_len={d.architecture['loop_len']:2}  "
        f"kozak_linker_len={d.architecture['kozak_linker_len']}"
    )

## 6. `evaluate_design()` — raw metrics and sequences, across the whole pool

**Raw** values only — no normalising, weighting or ranking here, that is
`engine.scoring`'s job. Keys are exactly the metric names `DEFAULT_V1` declares.

For the `"trailing"` layout specifically, `predicted_leakage`/`dynamic_range` are read
from the **toehold+stem region**, not the AUG — the AUG sits outside the hairpin here
and stays roughly accessible whether or not the trigger is bound, so AUG-region
accessibility would not discriminate ON from OFF for this layout (see
`evaluate_design`'s docstring). This is a new, unreviewed proxy — not a port of
anything previously validated.

Because `PAYLOAD_CDS` is set (cell 4), every design below is folded **with the real
effector gene's own first `PAYLOAD_HEAD_LENGTH` nucleotides**, not a placeholder — the
molecule `gate_folding_energy`/`predicted_leakage`/`dynamic_range` are measured on is
the one a ribosome would actually encounter once this gene is attached. Set
`PAYLOAD_CDS = None` in cell 4 and re-run to see how the numbers shift for the exact
same designs without that information.

`ranked` below holds **every** design in the pool (hundreds, from 150 triggers), best
first by `dynamic_range` — the cell only *prints* the top `TOP_DISPLAY`, but nothing
downstream (the report, cell 7's follow-ups) is limited to that display slice.

In [ ]:
all_metrics = [(d, gate.evaluate_design(d)) for d in designs]
ranked = sorted(all_metrics, key=lambda pair: pair[1]["dynamic_range"], reverse=True)

TOP_DISPLAY = 20
print(f"top {TOP_DISPLAY} of {len(ranked)}:\n")
print(
    f"{'trigger':>18}  {'region':>6}  {'loop_len':>8}  {'linker_len':>10}  {'leakage':>8}  "
    f"{'dyn_range':>9}  {'folding_energy':>14}"
)
for d, m in ranked[:TOP_DISPLAY]:
    region = "utr" if d.trigger_set.activators[0].start_index >= utr_start else "cds"
    print(
        f"{d.trigger_set.activators[0].trigger_id:>18}  {region:>6}  "
        f"{d.architecture['loop_len']:>8}  {d.architecture['kozak_linker_len']:>10}  "
        f"{m['predicted_leakage']:>8.3f}  {m['dynamic_range']:>9.3f}  "
        f"{m['gate_folding_energy']:>14.1f}"
    )

print(f"\nsequences, same order, top {TOP_DISPLAY} (best first):\n")
for d, _m in ranked[:TOP_DISPLAY]:
    print(f"{d.design_id}  ({d.length} nt)")
    print(f"  {gate.emit_sequence(d)}")

# Not this gate's job to rank in a real run (engine.scoring owns that) — but picking the
# top-ranked one here to carry through the rest of this notebook's cells.
design, metrics = ranked[0]
print(f"\nbest by dynamic_range: {design.design_id}")
for name, value in metrics.items():
    print(f"  {name:22} {value}")

Three things worth noticing:

1. **A design finally clears `dynamic_range` 1.0.** The best of the 904,
   `eukaryotic_toehold-trig-AREG-563-30-12-trailing-8-0`, scores 1.548 with
   `predicted_leakage` of just 0.018 — a real, working switch by this proxy, not
   another below-1.0 result to explain away. Widening the search from 5 triggers to
   150 is what found it; it wasn't visible in the earlier, smaller pool.
2. **The best design came from the CDS region, not the 3&#8242; UTR**, and every one of
   the top 20 does too — the first 3&#8242; UTR-derived design lands at **rank 78** of
   904 (`dynamic_range` 0.109). The 41-candidate UTR floor (cell 7) guarantees 3&#8242;
   UTR triggers get a fair *chance* to compete; it doesn't guarantee they win, and for
   this transcript and mechanism they mostly don't. That's a real result about *AREG*
   specifically, not a flaw in the quota.
3. **Pool composition**: 904 designs total, 300 tracing back to a 3&#8242; UTR trigger
   and 604 to a CDS-region one (cell 13) — roughly proportional to the 41/109 trigger
   split, since every compatible trigger yields the same 4 designs (`loop_len` x
   `kozak_linker_len` for `toehold_length=12`; the 36 nt triggers also unlock
   `toehold_length` 15 in some cases, mattering little here).

## 7. `emit_sequence()` — the synthesis-ready sequence

In [ ]:
print(gate.emit_sequence(design))

The Kozak element and start codon aren't visually obvious in that raw string — for this
`"trailing"` layout they sit right after the closed hairpin, not inside it (contrast
with `"loop"`, where they'd be buried in the middle). Note also what comes right after
the AUG here: the real effector gene's own first `PAYLOAD_HEAD_LENGTH` nucleotides
(`eff` from `CERNAL_FUNCTIONS.py`, set as `PAYLOAD_CDS` in cell 4), not a placeholder
— `"trailing"` carries no trailing linker of its own (commit `ee2f5f6`); cap-dependent
scanning initiates the instant the 40S subunit meets Kozak+AUG, so nothing after the
start codon plays any role in finding it, and whatever comes after is either the
payload (when known, as folded in here) or nothing at all (commit `74bf7b4`). Locate
Kozak and the start codon explicitly using `design.architecture` (which records
`aug_index` exactly) and the gate's own `KOZAK_EUKARYOTIC` constant:

In [ ]:
seq = design.sequence
aug_index = design.architecture["aug_index"]
kozak_index = seq.find(gate.KOZAK_EUKARYOTIC)
assert kozak_index != -1, "Kozak element not found — architecture assumptions above are stale"

marks = [" "] * len(seq)
for i in range(kozak_index, kozak_index + len(gate.KOZAK_EUKARYOTIC)):
    marks[i] = "K"
for i in range(aug_index, aug_index + 3):
    marks[i] = "A"

print(seq)
print("".join(marks), " K = Kozak (GCCACC)   A = start codon (AUG)")

## 8. Report — top 15, with a 3&#8242; UTR floor and 2D structure

`ReportBuilder`/`StructureRenderer` (`src/engine/stages/reporting.py`, stage 6/S14)
are still stubs — `render_structure`, `render_circuit` and `build` all raise
`NotImplementedError("Step 5")`. This section is **not** that; it's a report built
here in the notebook from the real `ranked` pool computed above, not a pipeline
capability.

`ranked` is already best-first by `dynamic_range`. Taking the top 15 outright could
easily land all 15 in the CDS region (it very nearly does — see cell 18's finding
that the first 3&#8242; UTR design sits at rank 78/904), so the same reserved-floor
approach from trigger selection (cell 7) applies again here: keep the top 15,
but if fewer than `MIN_UTR_IN_REPORT` of them are 3&#8242; UTR-derived, swap in the
best-ranked 3&#8242; UTR designs not already present, evicting the *worst*-ranked
non-UTR entries to make room — never touching rank #1.

In [ ]:
def select_with_region_floor(ranked_pool, n, min_from_region, in_region):
    """Top `n` of `ranked_pool` (already best-first), with at least `min_from_region`
    satisfying `in_region`. If the natural top `n` doesn't have enough, swap in the
    best-ranked region matches not already present, evicting the worst-ranked
    non-matching entries — never the entries that were already in-region."""
    top = list(ranked_pool[:n])
    have = sum(1 for d, m in top if in_region(d))
    if have >= min_from_region:
        return top

    top_ids = {d.design_id for d, m in top}
    region_reserve = [
        pair for pair in ranked_pool if in_region(pair[0]) and pair[0].design_id not in top_ids
    ]
    non_region_in_top = [pair for pair in top if not in_region(pair[0])]

    for extra in region_reserve[: min_from_region - have]:
        if not non_region_in_top:
            break
        evict = non_region_in_top.pop()  # worst-ranked non-region entry (list end)
        top.remove(evict)
        top.append(extra)

    top.sort(key=lambda pair: pair[1]["dynamic_range"], reverse=True)
    return top


def in_3utr(design):
    return design.trigger_set.activators[0].start_index >= utr_start


MIN_UTR_IN_REPORT = 3
REPORT_SIZE = 15
report = select_with_region_floor(ranked, REPORT_SIZE, MIN_UTR_IN_REPORT, in_3utr)

n_utr_in_report = sum(1 for d, m in report if in_3utr(d))
print(f"report: {len(report)} design(s), {n_utr_in_report} from the 3' UTR "
      f"(floor was {MIN_UTR_IN_REPORT})\n")

# Ensemble free energy (FoldEngine.partition — the partition-function total over
# every structure, not just the most stable one) rather than evaluate_design's own
# gate_folding_energy (MFE): "MFE reports a single structure, but RNA in solution
# occupies a distribution" (FoldEngine.partition's own docstring). This doesn't
# change the engine's scored metric anywhere — it's a display choice for this report
# table only.
print(
    f"{'#':>3}  {'trigger':>18}  {'region':>6}  {'geometry (nt)':>14}  "
    f"{'leakage':>8}  {'dyn_range':>9}  {'ensemble ΔG':>12}  {'gc%':>6}"
)
for rank, (d, m) in enumerate(report, 1):
    region = "utr" if in_3utr(d) else "cds"
    arch = d.architecture
    geometry = f"{arch['toehold_length']}/{arch['loop_len']}/{arch['kozak_linker_len']}"
    ensemble_energy = gate.folder.partition(d.sequence)
    print(
        f"{rank:>3}  {d.trigger_set.activators[0].trigger_id:>18}  {region:>6}  "
        f"{geometry:>14}  {m['predicted_leakage']:>8.3f}  {m['dynamic_range']:>9.3f}  "
        f"{ensemble_energy:>12.1f}  {m['gc_content']:>6.1f}"
    )

### 2D structure

`RNA.get_xy_coordinates` (ViennaRNA's own layout algorithm, not a house rule
violation — this is display-only, not a scientific computation, and this notebook
isn't `src/engine/`, so the "only two modules import RNA" rule (`CLAUDE.md` §5)
doesn't apply here) turns a dot-bracket string into 2D coordinates. Each base is
coloured by identity; Kozak and the start codon are outlined, and the 5&#8242;/3&#8242;
ends are labelled, so the mechanism this whole layout exists for is visible at a
glance.

**Tried and reverted: the ensemble's centroid structure (`fc.centroid()`).** A
centroid fold is the more principled choice for "the" ensemble-representative
structure — but calling it in a loop of 15, in the same process as the 904-design
pipeline above, segfaults **reliably** (isolated by bisection: `.mfe()` alone in a
loop here is fine, `.pf()` alone in a loop is fine, `.centroid()` alone in a fresh
process is fine — only `.centroid()` *after* the heavy pipeline, in this process,
crashes). That's a real ViennaRNA Python-binding memory instability under heavy
cumulative `fold_compound` churn, not a bug in this notebook's logic — but not
something to ship a notebook a teammate might run into. **This cell therefore plots
`gate.folder.mfe(seq).structure` instead** — the single lowest-energy structure,
via the same injected, cached `FoldEngine` already used everywhere else, proven
stable at this scale. It's a real predicted fold, still not `design.dot_bracket`
(the generator's idealized target, identical in shape for every design of matching
geometry regardless of how well it actually folds) — just not the ensemble centroid
specifically. `ensemble ΔG` in the table above (cell 7) still comes from `.pf()`
(via `FoldEngine.partition`), which is stable at this scale and unaffected by this.

The layout is computed for the **switch alone**, up to and including the start
codon — the attached payload head is real for every metric elsewhere in this
notebook, but a long unpaired tail made Kozak/AUG curl back toward the middle of the
picture instead of sitting at the 3&#8242; end where they belong, so it's dropped here
for the diagram specifically.

In [ ]:
from io import BytesIO

import RNA
import matplotlib.pyplot as plt
from IPython.display import Image, display

BASE_COLORS = {"A": "#FFD6A5", "U": "#A0C4FF", "G": "#FFADAD", "C": "#CAFFBF"}


def plot_structure_mini(ax, design, label):
    aug_index = design.architecture["aug_index"]
    kozak_index = design.sequence.find(gate.KOZAK_EUKARYOTIC)

    # Switch only, up to and including the start codon — see the markdown above for
    # why (Kozak/AUG land at the 3' end instead of curling into the middle) and why
    # this uses gate.folder.mfe() rather than a raw fc.centroid() call (a real,
    # reproducible ViennaRNA segfault at this scale, isolated and documented above).
    seq = design.sequence[: aug_index + 3]
    structure = gate.folder.mfe(seq).structure

    coords = RNA.get_xy_coordinates(structure)
    xs = [coords.get(i).X for i in range(len(structure))]
    ys = [coords.get(i).Y for i in range(len(structure))]

    ax.plot(xs, ys, "-", color="#B9C2BC", linewidth=1.1, zorder=1)

    stack = []
    for i, c in enumerate(structure):
        if c == "(":
            stack.append(i)
        elif c == ")":
            j = stack.pop()
            ax.plot(
                [xs[i], xs[j]], [ys[i], ys[j]], "-", color="#8899AA",
                linewidth=0.9, zorder=1, alpha=0.7,
            )

    highlight = set(range(kozak_index, kozak_index + len(gate.KOZAK_EUKARYOTIC)))
    highlight |= set(range(aug_index, aug_index + 3))

    for i, base in enumerate(seq):
        is_hl = i in highlight
        ax.scatter(
            xs[i], ys[i],
            s=26 if is_hl else 14,
            color=BASE_COLORS.get(base, "#EEEEEE"),
            edgecolors="#1A2620" if is_hl else "#666666",
            linewidths=1.1 if is_hl else 0.3,
            zorder=3 if is_hl else 2,
        )

    ax.text(xs[0], ys[0] + 9, "5'", fontsize=7, ha="center", color="#345", zorder=4)
    ax.text(xs[-1], ys[-1] + 9, "3'", fontsize=7, ha="center", color="#345", zorder=4)

    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(label, fontsize=9, fontweight="bold", loc="left")


fig, axes = plt.subplots(3, 5, figsize=(18, 11))
for i, (d, _m) in enumerate(report):
    region = "UTR" if in_3utr(d) else "CDS"
    label = f"#{i + 1}  {d.trigger_set.activators[0].trigger_id}  ({region})"
    plot_structure_mini(axes[i // 5][i % 5], d, label)

fig.suptitle(
    "Top 15 trailing-Kozak toeholds against AREG — MFE fold, switch only "
    "(payload head omitted), colour = base identity, outlined = Kozak/AUG",
    fontsize=12,
)
plt.tight_layout()
# Render to PNG bytes and display explicitly instead of plt.show() — plt.show()
# opens a native GUI window under any non-inline backend (the kernel's OS-default
# unless %matplotlib inline has already run in *this* kernel session, which a
# stale/reconnected kernel won't have done). Same pattern as render_heatmap_png
# below, and it works regardless of backend or restart state.
_buf = BytesIO()
fig.savefig(_buf, format="png", facecolor="white")
plt.close(fig)
_buf.seek(0)
display(Image(data=_buf.read()))

Base colours: A `#FFD6A5` · U `#A0C4FF` · G `#FFADAD` · C `#CAFFBF` (matching the
convention already used in `CERNAL_FUNCTIONS.py`'s own `plot_rna_structure`).

The correlation with the metrics table is visible directly in these diagrams: #1
(`dynamic_range` 1.55) folds into one clean extended hairpin with Kozak/AUG exposed
in a small accessible loop of their own. The three 3&#8242; UTR designs pulled in to meet
`MIN_UTR_IN_REPORT` land at #13&#8211;#15 — the worst `dynamic_range` in this 15 — and
their folds show why: Kozak/AUG buried inside a branched, multi-junction structure
rather than sitting in a clean loop, visibly less reachable, which is exactly what a
low `dynamic_range` means for this proxy. This is the real predicted fold explaining
the ranking, not the ranking asserted and then illustrated after the fact.

## 9. Publish-ready web report, matching `switch_candidate_report_updated_1.9.26.docx`

Someone else on the team was asked for a report shaped like a reference document
(`var/reports/switch_candidate_report_updated_1.9.26.docx`, for a different gate — an
antisense translational NOT-gate against an mCherry CDS). This section builds a report
generator that mirrors that document's structure as closely as sense allows, using this
notebook's own trailing-Kozak toehold data — the only report format this notebook
produces (an earlier, independently-styled generator, `render_report_html`, lived here
too; it was dropped in favour of standardising on this one).

**What carries over directly** (same shape, our data): the six-section structure (About /
Filtering / Ranking / Glossary / Summary / Detail), the per-candidate segmented "switch
layout" bar, the domain-by-domain sequence breakdown, and the "designed antiparallel
register" alignment figure.

**What had to change, and why:**

- **Opposite polarity.** The reference switch is a NOT-gate: translated by default,
  *silenced* when the trigger appears. This notebook's switch is the opposite — silent by
  default (the trailing-Kozak hairpin blocks scanning), *activated* when the trigger opens
  it (`docs/modalities.md`'s cap-dependent scanning-blockage mechanism, not steric RBS
  occlusion). "OFF" and "ON" are swapped between the two documents on purpose, not a typo.
- **Different metric vocabulary.** The reference computes `p_bound_area`, `off_run_frac`,
  `on_frac` and genome-relative openness percentiles from 500-sample Boltzmann ensembles
  against a real *E. coli* expression reference. This notebook's gate (`ToeholdGate.
  evaluate_design`) emits the nine `DEFAULT_V1` metric names — `predicted_leakage`,
  `dynamic_range`, `gate_folding_energy`, `trigger_accessibility`, `gc_content` — read
  from exact partition-function base-pair probabilities (`FoldEngine.
  base_pair_probabilities`), not resampled here. Different chemistry, different engine
  stage, genuinely different numbers — this report shows our own, not a forced remap onto
  the reference's names.
- **No RBS/spacer sandwiched mid-footprint.** The reference's RBS and AUG sit *inside* the
  trigger-paired footprint (steric occlusion). This design's Kozak and AUG trail *after*
  the whole closed hairpin (see `docs/modalities.md` and `toehold.py`'s own construction
  notes) — so the antiparallel register figure below shows them past the paired region
  entirely, not bracketed inside it, and the "Switch layout" bar has no `RBS`/`spacer`
  segments the reference's does.

In [ ]:
import base64
from io import BytesIO

import matplotlib.pyplot as plt


def _domain_offsets(design):
    """(name -> (start, end)) for every domain of a trailing-layout switch, in the same
    5'->3' order generate_designs assembles them. offsets["aug"][0] must equal
    design.architecture["aug_index"] — the two are computed independently and this is
    the cross-check that they still agree."""
    arch = design.architecture
    offsets = {}
    pos = 0

    def add(name, length):
        nonlocal pos
        offsets[name] = (pos, pos + length)
        pos += length

    add("leader", arch["leader_len"])
    add("toehold", arch["toehold_length"])
    add("pre_bulge", arch["stem_pre_bulge_len"])
    add("bulge", 3)
    add("post_bulge", arch["stem_post_bulge_len"])
    add("loop", arch["loop_len"])
    add("post_bulge_rc", arch["stem_post_bulge_len"])
    add("bulge_rc", 3)
    add("pre_bulge_rc", arch["stem_pre_bulge_len"])
    add("kozak_linker", arch["kozak_linker_len"])
    add("kozak", len(gate.KOZAK_EUKARYOTIC))
    add("aug", 3)
    assert offsets["aug"][0] == arch["aug_index"], "domain offsets drifted from aug_index"
    return offsets


def _png_from_fig(fig):
    buf = BytesIO()
    fig.savefig(buf, format="png", facecolor="white")
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("ascii")


def _structure_png(seq, structure, title, energy=None, highlight_ranges=None, figsize=(3.6, 3.2)):
    """One folded structure as a base64 PNG — the same RNA.get_xy_coordinates approach
    as the top-15 structure grid above, factored out here so this report doesn't reach
    into that cell's function (kept independent per this section's own intro note)."""
    coords = RNA.get_xy_coordinates(structure)
    xs = [coords.get(i).X for i in range(len(structure))]
    ys = [coords.get(i).Y for i in range(len(structure))]

    fig, ax = plt.subplots(figsize=figsize, dpi=130)
    ax.plot(xs, ys, "-", color="#B9C2BC", linewidth=1.1, zorder=1)

    stack = []
    for i, c in enumerate(structure):
        if c == "(":
            stack.append(i)
        elif c == ")":
            j = stack.pop()
            ax.plot(
                [xs[i], xs[j]], [ys[i], ys[j]], "-", color="#8899AA",
                linewidth=0.9, zorder=1, alpha=0.7,
            )

    highlight_ranges = highlight_ranges or []
    for i, _base in enumerate(seq):
        ring_color = next(
            (color for start, end, color in highlight_ranges if start <= i < end), None
        )
        ax.scatter(
            xs[i], ys[i],
            s=24 if ring_color else 13,
            color="#C7D0CA",  # plain base fill — no per-base colour, only highlight_ranges mark anything
            edgecolors=ring_color or "#666666",
            linewidths=1.3 if ring_color else 0.3,
            zorder=3 if ring_color else 2,
        )

    ax.text(xs[0], ys[0] + 8, "5'", fontsize=7, ha="center", color="#345")
    ax.text(xs[-1], ys[-1] + 8, "3'", fontsize=7, ha="center", color="#345")

    label = title if energy is None else f"{title} ({energy:.1f} kcal/mol)"
    ax.set_title(label, fontsize=9, fontweight="bold")
    ax.set_aspect("equal")
    ax.axis("off")
    plt.tight_layout()
    return _png_from_fig(fig)


def render_layout_bar_png(design):
    """Segmented layout bar, modeled on the reference report's "Switch layout" figure —
    one labeled, colored segment per domain of the trailing-Kozak switch."""
    arch = design.architecture
    segments = [
        ("toehold", arch["toehold_length"], "#4C6E8C"),
        ("pre-bulge", arch["stem_pre_bulge_len"], "#7E97AC"),
        ("bulge", 3, "#B7C4BC"),
        ("post-bulge", arch["stem_post_bulge_len"], "#7E97AC"),
        ("loop", arch["loop_len"], "#C7D0CA"),
        ("post-bulge", arch["stem_post_bulge_len"], "#7E97AC"),
        ("bulge", 3, "#B7C4BC"),
        ("pre-bulge", arch["stem_pre_bulge_len"], "#7E97AC"),
    ]
    if arch["kozak_linker_len"]:
        segments.append(("linker", arch["kozak_linker_len"], "#DCE4D8"))
    segments.append(("Kozak", len(gate.KOZAK_EUKARYOTIC), "#2E9E56"))
    segments.append(("AUG", 3, "#B5760E"))
    # Stops at the start codon, same scope as the reference report's own layout bar
    # (which stops at the post-AUG arm, before its reporter fusion) — the payload is
    # attached later at plasmid assembly, not part of this gate's own construction.

    total = sum(length for _, length, _ in segments)
    fig, ax = plt.subplots(figsize=(6.6, 1.5), dpi=130)
    x = 0
    for name, length, color in segments:
        ax.barh(0, length, left=x, height=0.6, color=color, edgecolor="white", linewidth=1.5)
        if length / total > 0.035:
            ax.text(
                x + length / 2, 0, str(length), ha="center", va="center",
                fontsize=8.5, fontweight="bold", color="white",
            )
            ax.text(
                x + length / 2, -0.55, name, ha="center", va="top",
                fontsize=7.5, color="#46524B",
            )
        x += length

    ax.set_xlim(0, total)
    ax.set_ylim(-1.15, 0.5)
    ax.axis("off")
    ax.set_title("Switch layout", fontsize=10, fontweight="bold", loc="left")
    plt.tight_layout()
    return _png_from_fig(fig)


def render_antiparallel_register_png(design):
    """Switch (5'->3') vs. the trigger footprint it was reverse-complemented from
    (3'->5'), aligned base-for-base with pairing lines — modeled on the reference
    report's "Designed antiparallel register" figure. Only the footprint pairs: the
    fixed leader before it and the Kozak/AUG after it have no trigger counterpart,
    same as the reference diagram's own gaps around its RBS/AUG."""
    arch = design.architecture
    aug_index = arch["aug_index"]
    switch_only = design.sequence[: aug_index + 3]
    trigger_sequence = design.trigger_set.activators[0].sequence

    leader_len = arch["leader_len"]
    footprint_len = (
        arch["toehold_length"] + arch["stem_pre_bulge_len"] + 3 + arch["stem_post_bulge_len"]
    )
    # binding_region = reverse_complement(trigger.sequence) in generate_designs, sliced
    # from its own start — so the footprint pairs with the trigger's *last*
    # footprint_len nt. Reversing (not complementing) that slice lines it up 3'->5'
    # beneath the switch's 5'->3' row, one complementary base per column.
    trigger_tail_reversed = trigger_sequence[-footprint_len:][::-1]

    n = len(switch_only)
    fig, ax = plt.subplots(figsize=(max(6.4, n * 0.115), 2.7), dpi=130)

    for i, base in enumerate(switch_only):
        ax.text(i, 1, base, ha="center", va="center", fontsize=7.5, family="monospace", fontweight="bold")
    for j, base in enumerate(trigger_tail_reversed):
        i = leader_len + j
        ax.text(i, -1, base, ha="center", va="center", fontsize=7.5, family="monospace", color="#46524B")
        ax.plot([i, i], [0.7, -0.7], color="#4C6E8C", linewidth=1.0, zorder=0)

    kozak_index = switch_only.find(gate.KOZAK_EUKARYOTIC)
    if kozak_index != -1:
        ax.plot(
            [kozak_index, kozak_index + len(gate.KOZAK_EUKARYOTIC) - 1], [1.4, 1.4],
            color="#2E9E56", linewidth=2,
        )
        ax.text(
            kozak_index + (len(gate.KOZAK_EUKARYOTIC) - 1) / 2, 1.6, "Kozak",
            ha="center", fontsize=7, color="#2E9E56", fontweight="bold",
        )
    ax.plot([aug_index, aug_index + 2], [1.4, 1.4], color="#B5760E", linewidth=2)
    ax.text(aug_index + 1, 1.6, "AUG", ha="center", fontsize=7, color="#B5760E", fontweight="bold")

    ax.text(-1.5, 1, "5'", ha="right", va="center", fontsize=7.5, color="#345")
    ax.text(n, 1, "3'", ha="left", va="center", fontsize=7.5, color="#345")
    ax.text(leader_len - 1.5, -1, "3'", ha="right", va="center", fontsize=7.5, color="#345")
    ax.text(leader_len + footprint_len - 1 + 1.5, -1, "5'", ha="left", va="center", fontsize=7.5, color="#345")

    ax.set_xlim(-3, n + 3)
    ax.set_ylim(-2, 2.1)
    ax.axis("off")
    ax.set_title(
        "Designed antiparallel register (switch vs. trigger footprint)",
        fontsize=9, fontweight="bold", loc="left",
    )
    plt.tight_layout()
    return _png_from_fig(fig)


# Smoke-test all three helpers on the top design before generating the full report below.
_test_design = report[0][0]
_bar = render_layout_bar_png(_test_design)
_register = render_antiparallel_register_png(_test_design)
_offsets = _domain_offsets(_test_design)
print(f"layout bar PNG: {len(_bar):,} b64 chars")
print(f"antiparallel register PNG: {len(_register):,} b64 chars")
print(f"domain offsets check out: aug at {_offsets['aug']}")


In [ ]:
import html as _html2
from datetime import datetime

_SCR_CSS = """
  :root {
    --scr-ground: #F6F8F4; --scr-surface: #FFFFFF; --scr-surface-2: #EEF2EA;
    --scr-ink: #131A16; --scr-ink-2: #46524B; --scr-ink-3: #75837B; --scr-line: #DCE4D8;
    --scr-accent: #2E9E56; --scr-accent-glow: #E4F7EA; --scr-accent-ink: #146134;
    --scr-structural: #4C6E8C; --scr-structural-bg: #E9EFF4;
    --scr-amber: #B5760E; --scr-amber-bg: #FBF0DD;
    --scr-shadow: 0 1px 2px rgba(19,26,22,0.04), 0 8px 24px -12px rgba(19,26,22,0.12);
    --scr-pill-utr-bg: #FBF0DD; --scr-pill-utr-ink: #8A5A0A;
    --scr-pill-cds-bg: #E9EFF4; --scr-pill-cds-ink: #3E5A73;
  }
  @media (prefers-color-scheme: dark) {
    :root:not([data-theme="light"]) {
      --scr-ground: #0E1512; --scr-surface: #141D18; --scr-surface-2: #1A2620;
      --scr-ink: #E9F2EC; --scr-ink-2: #AEC0B6; --scr-ink-3: #7C8D83; --scr-line: #26362D;
      --scr-accent: #4CE07A; --scr-accent-glow: #163524; --scr-accent-ink: #7DF0A2;
      --scr-structural: #8FB3D6; --scr-structural-bg: #1B2A38;
      --scr-amber: #E0AA4C; --scr-amber-bg: #362B14;
      --scr-shadow: 0 1px 2px rgba(0,0,0,0.3), 0 8px 24px -12px rgba(0,0,0,0.5);
      --scr-pill-utr-bg: #362B14; --scr-pill-utr-ink: #E0AA4C;
      --scr-pill-cds-bg: #1B2A38; --scr-pill-cds-ink: #8FB3D6;
    }
  }
  :root[data-theme="dark"] {
    --scr-ground: #0E1512; --scr-surface: #141D18; --scr-surface-2: #1A2620;
    --scr-ink: #E9F2EC; --scr-ink-2: #AEC0B6; --scr-ink-3: #7C8D83; --scr-line: #26362D;
    --scr-accent: #4CE07A; --scr-accent-glow: #163524; --scr-accent-ink: #7DF0A2;
    --scr-structural: #8FB3D6; --scr-structural-bg: #1B2A38;
    --scr-amber: #E0AA4C; --scr-amber-bg: #362B14;
    --scr-shadow: 0 1px 2px rgba(0,0,0,0.3), 0 8px 24px -12px rgba(0,0,0,0.5);
    --scr-pill-utr-bg: #362B14; --scr-pill-utr-ink: #E0AA4C;
    --scr-pill-cds-bg: #1B2A38; --scr-pill-cds-ink: #8FB3D6;
  }
  * { box-sizing: border-box; }
  body.scr { margin: 0; background: var(--scr-ground); color: var(--scr-ink); font-family: "IBM Plex Sans", system-ui, sans-serif; padding: 0 20px; }
  .scr-mono { font-family: "IBM Plex Mono", ui-monospace, monospace; font-variant-numeric: tabular-nums; }
  .scr-wrap { max-width: 980px; margin: 0 auto; padding-block: 44px 72px; }
  .scr-wrap h1 { font-size: clamp(26px, 4vw, 34px); font-weight: 700; line-height: 1.15; letter-spacing: -0.01em; margin: 0 0 8px; text-wrap: balance; }
  .scr-eyebrow { font-family: "IBM Plex Mono", monospace; font-size: 12px; letter-spacing: 0.08em; text-transform: uppercase; color: var(--scr-structural); margin: 0 0 10px; }
  .scr-dek { font-size: 15px; line-height: 1.6; color: var(--scr-ink-2); max-width: 72ch; margin: 0 0 32px; }
  .scr-wrap section { margin-bottom: 40px; }
  .scr-wrap h2 { font-size: 20px; font-weight: 700; letter-spacing: -0.01em; color: var(--scr-ink); margin: 0 0 12px; padding-bottom: 8px; border-bottom: 2px solid var(--scr-line); }
  .scr-wrap h3 { font-family: "IBM Plex Mono", monospace; font-size: 14px; font-weight: 600; color: var(--scr-accent-ink); background: var(--scr-accent-glow); display: inline-block; padding: 4px 10px; border-radius: 6px; margin: 0 0 14px; }
  .scr-p { font-size: 14px; line-height: 1.7; color: var(--scr-ink-2); margin: 0 0 12px; max-width: 78ch; }
  .scr-callout { background: var(--scr-structural-bg); border-left: 3px solid var(--scr-structural); border-radius: 4px; padding: 12px 16px; font-size: 13px; line-height: 1.6; color: var(--scr-ink-2); margin: 0 0 12px; }
  .scr-table-wrap { overflow-x: auto; border: 1px solid var(--scr-line); border-radius: 10px; background: var(--scr-surface); box-shadow: var(--scr-shadow); margin-bottom: 8px; }
  .scr-table-wrap table { width: 100%; border-collapse: collapse; font-size: 12.5px; }
  .scr-table-summary table { min-width: 640px; }
  .scr-table-wrap th { text-align: left; font-size: 10.5px; letter-spacing: 0.05em; text-transform: uppercase; color: var(--scr-ink-3); font-weight: 600; padding: 10px 12px; border-bottom: 1px solid var(--scr-line); background: var(--scr-surface-2); }
  .scr-table-wrap td { padding: 8px 12px; border-bottom: 1px solid var(--scr-line); color: var(--scr-ink-2); }
  .scr-table-wrap tr:last-child td { border-bottom: none; }
  .scr-table-wrap td.num { text-align: right; font-family: "IBM Plex Mono", monospace; }
  .scr-pill { display: inline-block; font-family: "IBM Plex Mono", monospace; font-size: 10px; font-weight: 600; letter-spacing: 0.03em; text-transform: uppercase; padding: 2px 6px; border-radius: 999px; }
  .scr-pill-utr { background: var(--scr-pill-utr-bg); color: var(--scr-pill-utr-ink); }
  .scr-pill-cds { background: var(--scr-pill-cds-bg); color: var(--scr-pill-cds-ink); }
  .scr-candidate { background: var(--scr-surface); border: 1px solid var(--scr-line); border-radius: 12px; padding: 20px 22px; box-shadow: var(--scr-shadow); margin-bottom: 18px; }
  .scr-candidate-head { display: flex; align-items: baseline; justify-content: space-between; gap: 12px; margin-bottom: 12px; flex-wrap: wrap; }
  .scr-candidate-head h3 { margin: 0; }
  .scr-candidate-id { font-family: "IBM Plex Mono", monospace; font-size: 12px; color: var(--scr-ink-3); }
  .scr-layout-img { max-width: 100%; height: auto; margin-bottom: 14px; }
  .scr-subgrid { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 14px; margin-bottom: 16px; }
  .scr-subtable-title { font-size: 11px; font-weight: 600; letter-spacing: 0.04em; text-transform: uppercase; color: var(--scr-ink-3); margin: 0 0 6px; }
  .scr-seq-block { font-size: 11.5px; line-height: 1.8; margin-bottom: 4px; }
  .scr-seq-block .label { display: inline-block; width: 90px; color: var(--scr-ink-3); font-family: "IBM Plex Mono", monospace; }
  .scr-seq-block .seq { color: var(--scr-ink); word-break: break-all; }
  .scr-structure-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap: 14px; margin-top: 10px; }
  .scr-structure-grid figure { margin: 0; background: var(--scr-surface-2); border-radius: 8px; padding: 6px; text-align: center; }
  .scr-structure-grid img { width: 100%; height: auto; }
  .scr-structure-grid figcaption { font-size: 11px; color: var(--scr-ink-3); margin-top: 4px; }
  .scr-register-img { max-width: 100%; height: auto; }
  footer.scr-footer { border-top: 1px solid var(--scr-line); padding-top: 18px; font-size: 11.5px; color: var(--scr-ink-3); }

  @media print {
    :root, :root:not([data-theme="light"]) {
      --scr-ground: #FFFFFF; --scr-surface: #FFFFFF; --scr-surface-2: #F3F5F1;
      --scr-ink: #131A16; --scr-ink-2: #3A443E; --scr-ink-3: #5C6660; --scr-line: #C9D2C4;
      --scr-accent: #1F7A42; --scr-accent-glow: #E4F7EA; --scr-accent-ink: #0F4A26;
      --scr-structural: #35526B; --scr-structural-bg: #E9EFF4;
      --scr-amber: #8A5A0A; --scr-amber-bg: #FBF0DD;
      --scr-shadow: none;
    }
    body.scr { padding: 0 12px; }
    .scr-candidate, .scr-table-wrap { box-shadow: none; }
    .scr-candidate-head, .scr-subgrid > *, .scr-seq-block, .scr-structure-grid figure, tr {
      break-inside: avoid; page-break-inside: avoid;
    }
    .scr-candidate { break-inside: auto; page-break-inside: auto; }
    section { break-inside: auto; }
    thead { display: table-header-group; }
  }
"""


def _scr_metric_table(rows):
    body = "".join(
        f'<tr><td>{_html2.escape(k)}</td><td class="num">{v}</td></tr>' for k, v in rows
    )
    return f'<div class="scr-table-wrap"><table><tbody>{body}</tbody></table></div>'


def render_switch_candidate_report_html(report_pairs, gate, transcript_gene="AREG"):
    """This notebook's report generator, modeled on the layout of
    `var/reports/switch_candidate_report_updated_1.9.26.docx` (a different gate — an
    antisense translational NOT-gate). See the markdown cell above this section for
    what carries over from the reference document and what had to change (opposite
    ON/OFF polarity, a different metric vocabulary, no RBS/AUG sandwiched
    mid-footprint)."""
    entries = []
    for rank, (d, m) in enumerate(report_pairs, 1):
        arch = d.architecture
        trigger = d.trigger_set.activators[0]
        aug_index = arch["aug_index"]
        switch_only = d.sequence[: aug_index + 3]
        offsets = _domain_offsets(d)

        off_energy = gate.folder.mfe(switch_only).energy
        complex_seq = f"{switch_only}&{trigger.sequence}"
        complex_energy = gate.folder.mfe(complex_seq).energy
        off_ensemble = gate.folder.partition(switch_only)
        on_ensemble = gate.folder.partition(complex_seq)

        trigger_structure = gate.folder.mfe(trigger.sequence).structure
        switch_structure = gate.folder.mfe(switch_only).structure
        complex_structure = gate.folder.mfe(complex_seq).structure

        kozak_start, _kozak_end = offsets["kozak"]
        _aug_start, aug_end = offsets["aug"]
        # The toehold + ascending stem is exactly binding_region[:footprint] in
        # generate_designs — the literal domain the trigger reverse-complements
        # against, i.e. "the place the trigger binds to" on the switch itself.
        footprint_start = offsets["toehold"][0]
        footprint_end = offsets["post_bulge"][1]
        KOZAK_AUG_COLOR = "#2E9E56"
        BINDING_COLOR = "#4C6E8C"  # matches the "toehold" segment colour in the layout bar

        entries.append({
            "rank": rank,
            "design": d,
            "metrics": m,
            "region": "utr" if in_3utr(d) else "cds",
            "arch": arch,
            "offsets": offsets,
            "switch_only": switch_only,
            "trigger": trigger,
            "off_energy": off_energy,
            "complex_energy": complex_energy,
            "off_ensemble": off_ensemble,
            "on_ensemble": on_ensemble,
            "layout_bar": render_layout_bar_png(d),
            "register": render_antiparallel_register_png(d),
            "trigger_structure_png": _structure_png(
                trigger.sequence, trigger_structure, "Trigger alone, MFE", energy=gate.folder.mfe(trigger.sequence).energy,
            ),
            "switch_structure_png": _structure_png(
                switch_only, switch_structure, "Switch alone (OFF), MFE", energy=off_energy,
                highlight_ranges=[
                    (footprint_start, footprint_end, BINDING_COLOR),
                    (kozak_start, aug_end, KOZAK_AUG_COLOR),
                ],
            ),
            "complex_structure_png": _structure_png(
                switch_only + trigger.sequence, complex_structure, "Switch + trigger (ON), illustrative complex MFE",
                energy=complex_energy,
                highlight_ranges=[
                    (footprint_start, footprint_end, BINDING_COLOR),
                    (kozak_start, aug_end, KOZAK_AUG_COLOR),
                    (len(switch_only), len(switch_only) + len(trigger.sequence), BINDING_COLOR),
                ],
            ),
        })

    summary_rows = "".join(
        f"""<tr>
          <td>{e["rank"]}</td>
          <td class="scr-mono">{_html2.escape(e["trigger"].trigger_id)}</td>
          <td><span class="scr-pill scr-pill-{e["region"]}">{e["region"]}</span></td>
          <td class="num">{e["trigger"].length}</td>
          <td class="num scr-mono">{e["arch"]["toehold_length"]}/{e["arch"]["loop_len"]}/{e["arch"]["kozak_linker_len"]}</td>
          <td class="num">{e["metrics"]["predicted_leakage"]:.3f}</td>
          <td class="num">{e["metrics"]["dynamic_range"]:.3f}</td>
          <td class="num">{e["off_ensemble"]:.1f} / {e["on_ensemble"]:.1f}</td>
          <td class="num">{e["metrics"]["gc_content"]:.1f}</td>
        </tr>"""
        for e in entries
    )

    glossary_rows = "".join(
        f"<tr><td>{_html2.escape(name)}</td><td>{_html2.escape(desc)}</td></tr>"
        for name, desc in [
            ("Toehold / pre-bulge / bulge / post-bulge", "Nucleotide lengths of the four stem domains, in that order (e.g. \"12/9/3/6\") — together with the loop they form the closed hairpin that blocks scanning by default."),
            ("Loop, Kozak linker", "Free-length spacer domains this layout can tune (unlike the reference's fixed RBS/AUG spacing) — swept per design, not scored directly."),
            ("Region (CDS / UTR)", "Where the trigger's window starts on the transcript: inside/before the coding sequence, or in the 3' UTR (from 100 nt before the CDS ends to the transcript's end)."),
            ("predicted_leakage", "OFF-state (switch alone, no trigger) mean accessibility of the toehold+stem footprint — how often the blocking hairpin fails to stay formed with no trigger present. Lower is better; this is the leak proxy."),
            ("dynamic_range", "ON-state accessibility of that same footprint (switch + trigger, cofolded with '&') divided by the OFF-state value above — the fold-change in openness the trigger causes. Higher is better."),
            ("gate_folding_energy / OFF MFE", "MFE free energy (kcal/mol) of the switch alone — the closed hairpin's own stability."),
            ("Complex MFE (illustrative)", "MFE of switch+trigger cofolded as a dimer ('&'-joined, not concatenated — CLAUDE.md §6). Labeled 'illustrative' because the 2D projection below draws it as one continuous backbone; the two strands are not covalently joined in reality."),
            ("Ensemble ΔG (OFF / ON)", "Partition-function free energy (kcal/mol) of the whole Boltzmann ensemble — not just the single MFE structure — for the switch alone and for switch+trigger respectively."),
            ("trigger_accessibility", "The trigger's own self-structure openness, carried from trigger scoring (stage 2) and read here, not recomputed."),
            ("gc_content", "GC percentage of the full switch sequence (0-100)."),
        ]
    )

    candidate_sections = []
    for e in entries:
        d, m, arch, offsets = e["design"], e["metrics"], e["arch"], e["offsets"]
        sw = e["switch_only"]

        def seq_line(label, key, offsets=offsets, sw=sw):
            start, end = offsets[key]
            return (
                f'<div class="scr-seq-block scr-mono">'
                f'<span class="label">{_html2.escape(label)}</span>'
                f'<span class="seq">{sw[start:end]}</span></div>'
            )

        param_rows = [
            ("Trigger window", f"{e['trigger'].length} nt (start {e['trigger'].start_index})"),
            ("Toehold", f"{arch['toehold_length']} nt"),
            ("Stem (pre-bulge / bulge / post-bulge)", f"{arch['stem_pre_bulge_len']} / 3 / {arch['stem_post_bulge_len']} nt"),
            ("Loop", f"{arch['loop_len']} nt"),
            ("Kozak linker", f"{arch['kozak_linker_len']} nt"),
            ("Kozak (fixed)", f"{len(gate.KOZAK_EUKARYOTIC)} nt — {gate.KOZAK_EUKARYOTIC}"),
            ("AUG (fixed)", "3 nt"),
            ("Full switch length (to AUG)", f"{len(sw)} nt"),
        ]
        binding_rows = [
            ("dynamic_range", f"{m['dynamic_range']:.3f}"),
            ("Ensemble ΔG, ON (complex)", f"{e['on_ensemble']:.1f} kcal/mol"),
            ("Complex MFE (illustrative single structure)", f"{e['complex_energy']:.1f} kcal/mol"),
        ]
        off_rows = [
            ("predicted_leakage", f"{m['predicted_leakage']:.3f}"),
            ("Ensemble ΔG, OFF (switch alone)", f"{e['off_ensemble']:.1f} kcal/mol"),
            ("gate_folding_energy (OFF MFE)", f"{m['gate_folding_energy']:.1f} kcal/mol"),
        ]
        other_rows = [
            ("trigger_accessibility", f"{m['trigger_accessibility']:.3f}"),
            ("gc_content", f"{m['gc_content']:.1f} %"),
        ]

        candidate_sections.append(f"""
        <div class="scr-candidate">
          <div class="scr-candidate-head">
            <h3>Candidate #{e["rank"]}</h3>
            <span class="scr-candidate-id">{_html2.escape(d.design_id)} &middot; trigger {_html2.escape(e["trigger"].trigger_id)}
              <span class="scr-pill scr-pill-{e["region"]}">{e["region"]}</span></span>
          </div>
          <img class="scr-layout-img" src="data:image/png;base64,{e["layout_bar"]}" alt="Switch layout for candidate {e["rank"]}">
          <div class="scr-subgrid">
            <div>
              <p class="scr-subtable-title">Design parameters</p>
              {_scr_metric_table(param_rows)}
            </div>
            <div>
              <p class="scr-subtable-title">OFF (switch alone, baseline)</p>
              {_scr_metric_table(off_rows)}
              <p class="scr-subtable-title" style="margin-top:10px;">ON (switch + trigger, triggered)</p>
              {_scr_metric_table(binding_rows)}
            </div>
            <div>
              <p class="scr-subtable-title">Other metrics</p>
              {_scr_metric_table(other_rows)}
            </div>
          </div>
          <p class="scr-subtable-title">Sequences</p>
          {seq_line("leader", "leader")}
          {seq_line("toehold", "toehold")}
          {seq_line("pre-bulge", "pre_bulge")}
          {seq_line("bulge", "bulge")}
          {seq_line("post-bulge", "post_bulge")}
          {seq_line("loop", "loop")}
          {seq_line("kozak linker", "kozak_linker")}
          {seq_line("Kozak", "kozak")}
          {seq_line("AUG", "aug")}
          <div class="scr-seq-block scr-mono"><span class="label">trigger</span><span class="seq">{e["trigger"].sequence}</span></div>
          <p class="scr-subtable-title" style="margin-top:14px;">Structures</p>
          <div class="scr-structure-grid">
            <figure><img src="data:image/png;base64,{e["trigger_structure_png"]}" alt="Trigger alone structure"><figcaption>Trigger alone, MFE fold — the isolated window's own fold, not the same measurement as trigger_accessibility below (that one is context-aware, computed within the full transcript by TriggerScorer; the two can disagree).</figcaption></figure>
            <figure><img src="data:image/png;base64,{e["switch_structure_png"]}" alt="Switch alone structure"><figcaption>Switch alone / OFF state, MFE fold. Blue outline: the toehold+stem footprint the trigger binds. Green outline: Kozak+AUG.</figcaption></figure>
            <figure><img src="data:image/png;base64,{e["complex_structure_png"]}" alt="Complex structure"><figcaption>Switch + trigger / ON state — illustrative single MFE structure (see glossary).</figcaption></figure>
          </div>
          <img class="scr-register-img" src="data:image/png;base64,{e["register"]}" alt="Antiparallel register for candidate {e["rank"]}" style="margin-top:14px;">
        </div>""")

    n_utr = sum(1 for e in entries if e["region"] == "utr")
    generated_dt = datetime.now().strftime("%Y-%m-%d")

    return f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Switch Candidate Report — {_html2.escape(transcript_gene)}</title>
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:wght@400;600;700&family=IBM+Plex+Mono:wght@400;600&display=swap">
<style>{_SCR_CSS}</style>
</head>
<body class="scr">
<div class="scr-wrap">
  <header>
    <p class="scr-eyebrow">Eukaryotic Trailing-Kozak Toehold Switch</p>
    <h1>Switch Candidate Report — {_html2.escape(transcript_gene)}</h1>
    <p class="scr-dek">Top {len(entries)} candidates ({n_utr} from the 3' UTR) out of {len(report_pairs)} scored designs. Generated {generated_dt}. Layout modeled on <code>switch_candidate_report_updated_1.9.26.docx</code> — see the notebook section above this report for what carries over and what had to change for a different gate chemistry.</p>
  </header>

  <section>
    <h2>1. About This Switch</h2>
    <p class="scr-p">The design goal is a translational ON-switch: an engineered mRNA (the "switch") that is <strong>silent by default</strong> and is <strong>activated</strong> when a specific endogenous mRNA (the "trigger") is present. This is the opposite polarity of the reference document's antisense NOT-gate, which is on by default and needs the trigger to turn OFF.</p>
    <p class="scr-p">The trigger is a window (30 or 36 nt here) taken from the target transcript — {_html2.escape(transcript_gene)} — either its coding sequence or its 3' UTR. The switch's toehold and stem are built as the reverse complement of that window, closing into a hairpin that blocks the 40S ribosomal subunit's scanning (cap-dependent scanning blockage, not the steric RBS occlusion the reference design uses — <code>docs/modalities.md</code>).</p>
    <p class="scr-p">Kozak and the start codon are fixed sequences, placed <strong>after</strong> the whole closed hairpin rather than sandwiched inside it (the reference design's RBS/AUG sit mid-footprint). Nothing after the AUG plays a role in the ribosome finding it for this mechanism, so the loop length and the linker before Kozak are free parameters this design sweeps rather than fixes.</p>
  </section>

  <section>
    <h2>2. How Candidates Were Filtered</h2>
    <p class="scr-p">Every trigger window <code>TriggerScorer</code> kept was checked with <code>is_compatible</code>, then expanded into switch variants by <code>generate_designs</code> — sweeping the toehold length, the loop length, and the Kozak linker length (<code>TRAILING_LOOP_LENGTHS</code> &times; <code>KOZAK_LINKER_LENGTHS</code>), all folded with ViennaRNA's exact partition function (<code>FoldEngine.base_pair_probabilities</code>), not a resampled ensemble.</p>
    <p class="scr-p">Two hard filters apply at the scoring layer, from <code>DEFAULT_V1</code> (<code>engine/scoring/profiles.py</code>), not a module-level literal in this gate: <code>predicted_leakage</code> at most 0.85, and <code>state_separation</code> at least 0.5. This notebook's own candidate pool below is additionally capped to the top {N_TRIGGERS} triggers with a 3' UTR floor — a notebook-level slice of the pipeline, not a scoring-layer filter.</p>
  </section>

  <section>
    <h2>3. How Candidates Were Ranked</h2>
    <p class="scr-p">Within the scored pool, this report ranks by <code>dynamic_range</code> (the ON/OFF fold-change in footprint accessibility) descending, breaking ties on <code>predicted_leakage</code> ascending — the tie-break order <code>CLAUDE.md</code> §2 specifies for the shared hard-filter set. This is a demonstration ranking for this notebook; a real run's ranking is <code>engine.scoring</code>'s job (normalized, weighted across all nine <code>DEFAULT_V1</code> metrics), not this gate's.</p>
  </section>

  <section>
    <h2>4. Metric Glossary</h2>
    <div class="scr-table-wrap">
      <table><thead><tr><th>Term</th><th>Meaning</th></tr></thead>
      <tbody>{glossary_rows}</tbody></table>
    </div>
  </section>

  <section>
    <h2>5. Candidate Summary</h2>
    <div class="scr-table-wrap scr-table-summary">
      <table>
        <thead><tr>
          <th>#</th><th>Trigger</th><th>Region</th><th>Window (nt)</th><th>toehold/loop/linker</th>
          <th>leakage</th><th>dyn. range</th><th>ensemble ΔG off/on</th><th>gc%</th>
        </tr></thead>
        <tbody>{summary_rows}</tbody>
      </table>
    </div>
  </section>

  <section>
    <h2>6. Candidate Detail</h2>
    {"".join(candidate_sections)}
  </section>

  <footer class="scr-footer">
    Generated by <code>render_switch_candidate_report_html</code> in this notebook. ViennaRNA {gate.folder.versions()["ViennaRNA"]}.
  </footer>
</div>
</body>
</html>"""


print("render_switch_candidate_report_html defined")


In [ ]:
from pathlib import Path

scr_out_path = Path.cwd()
for _base in (scr_out_path, *scr_out_path.parents):
    if (_base / "pyproject.toml").exists():
        scr_out_path = _base / "var" / "reports" / "switch_candidate_report_areg.html"
        break

scr_out_path.parent.mkdir(parents=True, exist_ok=True)
scr_out_path.write_text(render_switch_candidate_report_html(report, gate, transcript_gene="AREG"))

print(f"wrote {scr_out_path.stat().st_size:,} bytes to {scr_out_path}")

import subprocess
import sys

scr_pdf_path = scr_out_path.with_suffix(".pdf")
scr_render_pdf_script = scr_out_path.parent
for _base in (Path.cwd(), *Path.cwd().parents):
    _candidate = _base / "src/engine/gates/notebooks/toehold/render_pdf.py"
    if _candidate.exists():
        scr_render_pdf_script = _candidate
        break

subprocess.run(
    [sys.executable, str(scr_render_pdf_script), str(scr_out_path), str(scr_pdf_path)],
    check=True,
)
print(f"\nTo publish this PDF: hand {scr_pdf_path} to whoever needs a static copy.")
print("To publish the HTML live: hand the .html path to Claude's Artifact tool.")

## Where this fits in a real run

This notebook now pools 904 designs across 150 triggers (41 from the 3&#8242; UTR, 109
from the CDS region) and one layout — closer to a real run, but still a slice of it.
In the actual pipeline:

- Every trigger `TriggerScorer` keeps (not just the top `N_TRIGGERS`) gets checked
  with `is_compatible` and, if it passes, run through `generate_designs` — and by
  default that sweeps **both** `"loop"` and `"trailing"` layouts (this notebook
  restricted to `"trailing"` only via `kozak_layouts` — see cell 4). No 3&#8242; UTR
  quota exists anywhere in `engine.stages.triggers` — the floor enforced here (cell 7)
  is a notebook-level policy, not an engine feature; if a real run needs this
  guarantee, it belongs in `Constraints` or `TriggerScorer` itself, not reimplemented
  per caller.
- Every design's raw metrics go through `engine.scoring` — `build_metrics`,
  `weighted_score`, `failed_filter`, `rank_candidates` — which is what actually
  decides which designs survive and how they rank, comparably with every other gate
  family's designs, every trigger, **and across layouts**. Neither this notebook nor
  the gate itself picks a winner — sorting by `dynamic_range` alone (cell 17) is a
  stand-in for that, not the real ranking.
- `CandidateStore` records provenance and writes the stage snapshot; nothing here
  hand-rolls a results CSV.
- The **payload** folds into evaluation when known (`PAYLOAD_CDS` in cell 4) but the
  *actual* fused construct is still assembled later, at plasmid assembly
  (`PlasmidBuilder.build(circuit, DesiredOutcome.CUSTOM, custom_payload=...)`), using
  the complete gene, not just its folded-in head.

The two-input AND version (`EukaryoticToeholdAndGate`) is **not** implemented yet —
its `generate_designs` still raises `NotImplementedError("Step 5")`. This notebook
only covers the single-input case.

Open questions this work surfaced, not resolved here (see commits `d754812`,
`ee2f5f6`, `74bf7b4`): whether `"loop"` should still ship as the eukaryotic default
now that `"trailing"` exists (and now that `"loop"`'s Kozak-AUG adjacency is known to
be broken); whether `TRAILING_LOOP_LENGTHS`/`KOZAK_LINKER_LENGTHS` are the right
ranges to sweep; whether the `"trailing"` leakage proxy (toehold+stem accessibility)
is the right measurement for scanning-ribosome blockage at all; whether fusing the
payload directly after the switch's own placeholder AUG (rather than dropping that AUG
in favour of the payload's own) is the right call; the CDS boundary here is a
longest-ORF heuristic, not real annotation — wrong for any transcript with a shorter
true CDS or a non-canonical start; and whether a 3&#8242; UTR floor belongs in
`Constraints` as a first-class, engine-level concept rather than a per-notebook
policy, given the trigger scoring map's own emphasis on where a trigger sits along the
transcript.